# RustPy-XlsxWriter Quick Start

This notebook demonstrates all key features of `rustpy_xlsxwriter` using the `FastExcel` builder class.

In [ ]:
# Install if needed
# !pip install rustpy-xlsxwriter pandas

## 1. Basic — Single Sheet

In [ ]:
from rustpy_xlsxwriter import FastExcel

records = [
    {"Name": "Alice", "Age": 30, "City": "New York"},
    {"Name": "Bob", "Age": 25, "City": "San Francisco"},
    {"Name": "Charlie", "Age": 35, "City": "Chicago"}
]

FastExcel("output_basic.xlsx").sheet("Employees", records).save()
print("✅ output_basic.xlsx created")

## 2. Multiple Sheets

In [ ]:
employees = [{"Name": "Alice", "Salary": 95000.50}, {"Name": "Bob", "Salary": 82000.75}]
inventory = [{"Product": "Laptop", "Price": 1200.00}, {"Product": "Phone", "Price": 800.00}]

(
    FastExcel("output_multi.xlsx")
    .sheet("Employees", employees)
    .sheet("Inventory", inventory)
    .save()
)
print("✅ output_multi.xlsx created with 2 sheets")

## 3. Pandas DataFrame

In [ ]:
import pandas as pd

df = pd.DataFrame({
    "Name": ["Alice", "Bob", "Charlie"],
    "Score": [95.678, 88.123, 72.456],
    "Passed": [True, True, False]
})
df

In [ ]:
# With float formatting + bold index columns
(
    FastExcel("output_dataframe.xlsx")
    .format(float_format="0.00", index_columns=["Name"])
    .sheet("Results", df)
    .save()
)
print("✅ output_dataframe.xlsx created (formatted + bold index)")

## 4. Freeze Panes

In [ ]:
data = [{"ID": i, "Name": f"Item {i}", "Value": i * 1.5} for i in range(100)]

# Freeze header row
FastExcel("output_frozen.xlsx").freeze(row=1).sheet("Data", data).save()
print("✅ output_frozen.xlsx created (frozen header row)")

# Per-sheet freeze
(
    FastExcel("output_freeze_custom.xlsx")
    .freeze(row=1)                              # all sheets
    .freeze(row=2, col=1, sheet="Summary")      # override
    .sheet("Data", data)
    .sheet("Summary", data[:10])
    .save()
)
print("✅ output_freeze_custom.xlsx created (per-sheet config)")

## 5. In-Memory Buffer (BytesIO)

In [ ]:
import io

buf = io.BytesIO()
FastExcel(buf).sheet("Sheet1", [{"Key": "value"}]).save()

xlsx_bytes = buf.getvalue()
print(f"✅ Generated {len(xlsx_bytes):,} bytes in memory")
print(f"   Magic bytes: {xlsx_bytes[:4]}  (PK = valid xlsx)")

## 6. Generator Streaming (Memory-Efficient)

In [ ]:
def generate_rows(n: int):
    """Yield rows lazily — never loads all data into memory."""
    for i in range(n):
        yield {"id": i, "name": f"user_{i}", "score": i * 0.1}

FastExcel("output_generator.xlsx").sheet("Data", generate_rows(100_000)).save()
print("✅ output_generator.xlsx created (100K rows via generator)")

## 7. Password Protection

In [ ]:
(
    FastExcel("output_protected.xlsx", password="s3cret!")
    .format(float_format="0.00")
    .freeze(row=1)
    .sheet("Accounts", [{"Account": "Savings", "Balance": 12500.50}])
    .save()
)
print("✅ output_protected.xlsx created (password: s3cret!)")

## 8. Full Featured — All Options Combined

In [ ]:
from datetime import datetime

employees = [
    {"ID": 1, "Name": "Alice", "Salary": 95000.50, "Hired": datetime(2021, 3, 15), "Active": True},
    {"ID": 2, "Name": "Bob", "Salary": 82000.75, "Hired": datetime(2022, 7, 1), "Active": True},
    {"ID": 3, "Name": "Charlie", "Salary": 110000.00, "Hired": datetime(2020, 1, 10), "Active": False},
]

departments = [
    {"Dept": "Engineering", "Budget": 500000.00, "Headcount": 25},
    {"Dept": "Marketing", "Budget": 200000.00, "Headcount": 12},
]

(
    FastExcel("output_full.xlsx", password="admin123")
    .format(float_format="0.00", index_columns=["ID", "Dept"])
    .freeze(row=1)
    .freeze(row=1, col=2, sheet="Employees")
    .sheet("Employees", employees)
    .sheet("Departments", departments)
    .save()
)
print("✅ output_full.xlsx created")
print("   - 2 sheets, password protected")
print("   - Float format, bold index columns")
print("   - Frozen headers + custom freeze on Employees")

## Cleanup

In [ ]:
import glob, os
for f in glob.glob("output_*.xlsx"):
    os.remove(f)
print("🧹 All output files cleaned up")